In [ ]:
# %pip restarts Python, keep before %run

In [ ]:
%pip install -q geopandas shapely folium

In [ ]:
%run ../data/globalvariables

In [ ]:
import geopandas as gpd
import folium
from folium.plugins import MarkerCluster

In [ ]:
# Load district polygons
distritos_pdf = spark.table(f"{GOLD_TABLE}.dim_distrito").toPandas()
distritos_gdf = gpd.GeoDataFrame(
    distritos_pdf,
    geometry=gpd.GeoSeries.from_wkt(distritos_pdf["geometry"]),
    crs="EPSG:4326",
)

In [ ]:
# Load stations and traffic points
estaciones_pdf = spark.table(f"{GOLD_TABLE}.dim_estacion_aire").toPandas()
puntos_pdf = spark.table(f"{GOLD_TABLE}.dim_punto_trafico").toPandas()

# Bronze CSVs land as strings without a schema contract
for pdf in (estaciones_pdf, puntos_pdf):
    pdf["longitud"] = pd.to_numeric(pdf["longitud"])
    pdf["latitud"] = pd.to_numeric(pdf["latitud"])

In [ ]:
# Center map on district average
center_lat = distritos_pdf["centroid_lat"].mean()
center_lon = distritos_pdf["centroid_lon"].mean()
mapa = folium.Map(location=[center_lat, center_lon], zoom_start=11, tiles="cartodbpositron")

In [ ]:
# District polygons colored by air coverage
def distrito_style(feature):
    cobertura = feature["properties"]["cobertura_aire"]
    return {
        "fillColor": "#2ca25f" if cobertura else "#bdbdbd",
        "color": "#525252",
        "weight": 1,
        "fillOpacity": 0.4,
    }

folium.GeoJson(
    distritos_gdf.__geo_interface__,
    name="Distritos",
    style_function=distrito_style,
    tooltip=folium.GeoJsonTooltip(
        fields=["nombre", "n_estaciones_aire", "n_puntos_trafico"],
        aliases=["Distrito", "Estaciones aire", "Puntos trafico"],
    ),
).add_to(mapa)

In [ ]:
# Air stations as blue markers
for _, row in estaciones_pdf.iterrows():
    folium.CircleMarker(
        location=[row["latitud"], row["longitud"]],
        radius=4, color="#08519c", fill=True, fill_opacity=0.9,
        tooltip=row["estacion"],
    ).add_to(mapa)

In [ ]:
# Traffic points as clustered red markers
cluster = MarkerCluster(name="Puntos trafico").add_to(mapa)
for _, row in puntos_pdf.iterrows():
    folium.CircleMarker(
        location=[row["latitud"], row["longitud"]],
        radius=3, color="#a50f15", fill=True, fill_opacity=0.7,
        tooltip=row["nombre"],
    ).add_to(cluster)

In [ ]:
# Render inline
folium.LayerControl().add_to(mapa)
mapa